In [1]:
import pandas as pd

df = pd.read_csv("data.csv")

In [2]:
df[['title', 'description', 'brand', 'categories', 'features']].head()

,title,description,brand,categories,features
0,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",Saucony,"[""Clothing, Shoes & Jewelry"",""Men"",""Shoes"",""At...",NaN
1,Kishigo Premium Black Series Heavy Duty Unisex...,The Kishigo Premium Black Series Heavy Duty Ve...,Kishigo,"[""Tools & Home Improvement"",""Safety & Security...",NaN
2,TWINSLUXES Solar Post Cap Lights Outdoor - Wat...,Solar Post Cap Lights Waterproof LED Fence Pos...,TWINSLUXES,"[""Tools & Home Improvement"",""Lighting & Ceilin...","[""QUICK AND EASY INSTALLATIONOur solar post ca..."
3,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,Accutire,"[""Automotive"",""Tools & Equipment"",""Tire & Whee...","[""Heavy duty construction and rugged design fo..."
4,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,SAURA LIFE SCIENCE,"[""Beauty"",""Hair Care"",""Hair Oils""]","[""BENEFITS. 1 Reduces hair fall 2 Boosts hair ..."


In [3]:
df[['title', 'description', 'brand', 'categories', 'features']].isnull().sum()

title            0
description      2
brand            1
categories       0
features       182
dtype: int64

In [4]:
df[['title', 'description', 'brand', 'categories', 'features']].nunique()

title          999
description    995
brand          894
categories     787
features       799
dtype: int64

In [5]:
df['recommendation_text'] = (
    df['title'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['brand'].fillna('') + ' ' +
    df['categories'].fillna('') + ' ' +
    df['features'].fillna('')
)

In [6]:
df[['title', 'recommendation_text']].head()

,title,recommendation_text
0,Saucony Men's Kinvara 13 Running Shoe,Saucony Men's Kinvara 13 Running Shoe When it ...
1,Kishigo Premium Black Series Heavy Duty Unisex...,Kishigo Premium Black Series Heavy Duty Unisex...
2,TWINSLUXES Solar Post Cap Lights Outdoor - Wat...,TWINSLUXES Solar Post Cap Lights Outdoor - Wat...
3,Accutire MS-4021B Digital Tire Pressure Gauge ...,Accutire MS-4021B Digital Tire Pressure Gauge ...
4,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...


In [7]:
df['recommendation_text'].str.len().describe()

count     1000.000000
mean      2436.091000
std       3197.865093
min        128.000000
25%        994.000000
50%       1841.500000
75%       2880.000000
max      27498.000000
Name: recommendation_text, dtype: float64

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df['recommendation_text'])

tfidf_matrix.shape

(1000, 5000)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

similarity.shape

(1000, 1000)

In [10]:
def recommend_products(product_index, top_n=5):
    scores = list(enumerate(similarity[product_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = scores[1:top_n+1]

    return df.iloc[[i[0] for i in recommendations]][
        ['title', 'brand', 'categories']
    ]

In [11]:
recommend_products(0)

,title,brand,categories
382,adidas Men's Ultraboost Personal Best Running ...,adidas,"[""Clothing, Shoes & Jewelry"",""Men"",""Shoes"",""At..."
244,adidas Men's Racer Tr21 Running Shoe,adidas,"[""Clothing, Shoes & Jewelry"",""Men"",""Shoes"",""At..."
516,ASICS Women's Gel-Cumulus 20 Running Shoes,Asics,"[""Clothing, Shoes & Jewelry"",""Women"",""Shoes"",""..."
275,Merrell Men's Trail Glove 6 Sneaker,Merrell,"[""Clothing, Shoes & Jewelry"",""Men"",""Shoes"",""At..."
253,Skechers Women's Max Cushioning Elite Sneaker,Skechers,"[""Clothing, Shoes & Jewelry"",""Women"",""Shoes"",""..."


In [12]:
print(df.iloc[0][['title', 'brand', 'categories']])

title                     Saucony Men's Kinvara 13 Running Shoe
brand                                                   Saucony
categories    ["Clothing, Shoes & Jewelry","Men","Shoes","At...
Name: 0, dtype: object


In [13]:
product_index = 0

print("Because you viewed:")
print(df.iloc[product_index]['title'])

print("\nYou may also like:")

recommendations = recommend_products(product_index, top_n=5)

for i, row in recommendations.iterrows():
    print(f"• {row['title']} | {row['brand']}")

Because you viewed:
Saucony Men's Kinvara 13 Running Shoe

You may also like:
• adidas Men's Ultraboost Personal Best Running Shoe | adidas
• adidas Men's Racer Tr21 Running Shoe | adidas
• ASICS Women's Gel-Cumulus 20 Running Shoes | Asics
• Merrell Men's Trail Glove 6 Sneaker | Merrell
• Skechers Women's Max Cushioning Elite Sneaker | Skechers


In [14]:
product_index = 0

scores = list(enumerate(similarity[product_index]))
scores = sorted(scores, key=lambda x: x[1], reverse=True)

for index, score in scores[1:6]:
    print(f"{score:.3f} | {df.iloc[index]['title']}")

0.536 | adidas Men's Ultraboost Personal Best Running Shoe
0.419 | adidas Men's Racer Tr21 Running Shoe
0.368 | ASICS Women's Gel-Cumulus 20 Running Shoes
0.340 | Merrell Men's Trail Glove 6 Sneaker
0.305 | Skechers Women's Max Cushioning Elite Sneaker
